# Thunders AI — Inference

This notebook demonstrates how to run inference using the Thunders AI framework.
We'll cover model loading, various inference modes, and performance benchmarking.

In [ ]:
# Install Thunders AI if needed
# !pip install thunders-ai[all]

import thunders_ai
from thunders_ai import ThundersAI
from thunders_ai.models import Model, InferenceConfig

print(f"Thunders AI version: {thunders_ai.__version__}")

## 1. Load Model

Load a pre-trained model for inference. Thunders AI supports local and remote models.

In [ ]:
# Initialize the Thunders AI client
ai = ThundersAI(device="cpu")  # Change to "cuda" for GPU

# Or load a specific model directly
model = Model.from_pretrained(
    "thunders-ai/default",
    device="cpu",
    dtype="float32",
)

print(f"Model loaded: {model.name}")
print(f"Parameters: {model.num_parameters:,}")
print(f"Device: {model.device}")

## 2. Text Inference (Chat)

Run text generation and chat completions.

In [ ]:
# Simple chat completion
response = ai.chat(
    "Explain the concept of attention mechanisms in transformers.",
    max_tokens=256,
    temperature=0.7,
)
print("Chat Response:")
print(response)
print()

# Streaming chat
print("Streaming Response:")
for chunk in ai.chat_stream("Write a short poem about AI.", max_tokens=128):
    print(chunk, end="", flush=True)
print()

## 3. Vision Inference

Run image analysis, classification, and object detection.

In [ ]:
# Image classification
result = ai.vision.classify("sample_image.jpg")
print(f"Classification: {result.label} (confidence: {result.confidence:.2%})")

# Object detection
detections = ai.vision.detect("sample_image.jpg", confidence_threshold=0.5)
for det in detections:
    print(f"  {det.label}: {det.score:.2%} at {det.bbox}")

# Image description
description = ai.vision.describe("sample_image.jpg")
print(f"\nDescription: {description}")

## 4. Speech Inference

Transcribe audio and synthesize speech.

In [ ]:
# Speech to text
transcript = ai.speech.transcribe("sample_audio.wav")
print(f"Transcript: {transcript.text}")
print(f"Language: {transcript.language}")
print(f"Duration: {transcript.duration:.1f}s")

# Text to speech
audio = ai.speech.synthesize(
    "Welcome to Thunders AI, your unified artificial intelligence platform.",
    voice="alloy",
    speed=1.0,
)
audio.save("tts_output.wav")
print(f"\nTTS output saved: tts_output.wav ({audio.duration:.1f}s)")

## 5. Batch Inference

Process multiple inputs efficiently with batch inference.

In [ ]:
# Batch text generation
prompts = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is deep learning?",
    "Describe reinforcement learning.",
]

batch_results = ai.chat_batch(
    prompts,
    max_tokens=100,
    temperature=0.5,
)

for prompt, result in zip(prompts, batch_results):
    print(f"Q: {prompt}")
    print(f"A: {result[:100]}...")
    print()

## 6. Benchmark Performance

Measure inference latency and throughput.

In [ ]:
import time
import json

# Benchmark configuration
num_warmup = 3
num_runs = 20
prompt = "Explain quantum computing in simple terms."

# Warmup runs
for _ in range(num_warmup):
    _ = ai.chat(prompt, max_tokens=64)

# Benchmark runs
latencies = []
for i in range(num_runs):
    start = time.perf_counter()
    _ = ai.chat(prompt, max_tokens=64)
    elapsed_ms = (time.perf_counter() - start) * 1000
    latencies.append(elapsed_ms)

# Compute statistics
latencies_sorted = sorted(latencies)
avg_ms = sum(latencies) / len(latencies)
p50 = latencies_sorted[len(latencies_sorted) // 2]
p95 = latencies_sorted[int(len(latencies_sorted) * 0.95)]
p99 = latencies_sorted[int(len(latencies_sorted) * 0.99)]

benchmark_results = {
    "model": "thunders-ai/default",
    "device": str(model.device),
    "num_runs": num_runs,
    "latency_ms": {
        "mean": round(avg_ms, 2),
        "p50": round(p50, 2),
        "p95": round(p95, 2),
        "p99": round(p99, 2),
        "min": round(min(latencies), 2),
        "max": round(max(latencies), 2),
    },
    "throughput_req_per_sec": round(1000 / avg_ms, 2),
}

print("Benchmark Results:")
print(json.dumps(benchmark_results, indent=2))

# Save results
with open("inference_benchmark.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)
print("\nResults saved to inference_benchmark.json")